# Encoder comparison: Patent-BERT vs SBERT models (GPU)
### SDTD Patent Intelligence - prior-art retrieval benchmark against real citations

Same test as `finetune/compare_encoders.py`, but fast on a Colab GPU, so the **whole** computer-science subset
(or the whole dataset) can be used.

For each query patent its title + abstract is embedded, the corpus is searched, and we check where the patents it
**actually cites** rank. Metrics for **SD** (cosine) and **SDTD** (top 100 re-ranked with (TD + 1) x SD / 2).

**Upload from the project folder when asked:**
1. `patents_meta.parquet` (project root)
2. `data/processed/eval_citations.parquet` (made by `python -m finetune.build_pairs`)

Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all.

In [ ]:
# Step 1: GPU check and dependencies
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q sentence-transformers faiss-cpu

In [ ]:
# Step 2: Settings
SUBSET = "cs"            # "cs" = CPC G06 / G16 / H04L patents, "all" = all 118,840 patents
CORPUS_SIZE = 10000      # same 10,000 CS patents as the laptop run (None = all 33,934 CS patents)
MAX_QUERIES = None       # None = every citing patent in the subset
POOL = 100               # candidates re-ranked by SDTD (best Recall@10 in the full benchmark)
SEED = 13
MODELS = [
    "patent-bert",                               # anferico/bert-for-patents, mean pooling (the app's encoder)
    # "AI-Growth-Lab/PatentSBERTa",              # already measured on the laptop (Recall@10 SDTD 0.296)
    "sentence-transformers/all-mpnet-base-v2",   # general-purpose SBERT
]
CS_PREFIXES = ("G06", "G16", "H04L")

In [ ]:
# Step 3: Load the data (upload, or put both files in /content beforehand)
import os
import pandas as pd

needed = ["patents_meta.parquet", "eval_citations.parquet"]
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    from google.colab import files
    print("Upload:", missing)
    files.upload()

meta = pd.read_parquet("patents_meta.parquet", columns=["patent_id", "meta", "codes"])
meta["codes"] = meta.codes.map(list)
ev = pd.read_parquet("eval_citations.parquet")
print(f"{len(meta):,} patents, {len(ev):,} citing patents in the citation benchmark")

In [ ]:
# Step 4: Build the test set
import random

if SUBSET == "cs":
    in_subset = meta.codes.map(lambda cs: any(str(c).startswith(CS_PREFIXES) for c in cs))
else:
    in_subset = pd.Series(True, index=meta.index)
subset_ids = set(meta.patent_id[in_subset])

queries = []
for q, rel in zip(ev.query_id, ev.relevant_ids):
    rel = [r for r in rel if r in subset_ids]
    if q in subset_ids and rel:
        queries.append((q, set(rel)))
if MAX_QUERIES:
    queries = queries[:MAX_QUERIES]

must_have = {q for q, _ in queries} | {r for _, rel in queries for r in rel}
others = sorted(subset_ids - must_have)
random.Random(SEED).shuffle(others)
size = CORPUS_SIZE or len(subset_ids)
corpus_ids = sorted(must_have) + others[: max(size - len(must_have), 0)]

sub = meta.set_index("patent_id").loc[corpus_ids].reset_index()
print(f"Test set ({SUBSET}): {len(sub):,} patents searched, {len(queries)} query patents, "
      f"{sum(len(r) for _, r in queries):,} citations")

In [ ]:
# Step 5: SDTD (same hierarchical CPC/IPC similarity as agents/analyst_agent.py) and metrics
import re
import numpy as np

IPC_LEVEL_WEIGHTS = {0: 0.0, 1: 0.25, 2: 0.5, 3: 0.75, 4: 0.9, 5: 1.0}
_IPC_RE = re.compile(r"^([A-HY])(\d{2})?([A-Z])?(\d{1,4})?(?:/(\d{1,6}))?")

def ipc_levels(code):
    code = re.sub(r"\s+", "", str(code).upper())
    m = _IPC_RE.match(code)
    if not m:
        return None
    levels, prefix = [], ""
    for part, sep in zip(m.groups(), ("", "", "", "", "/")):
        if part is None:
            break
        prefix += sep + part
        levels.append(prefix)
    return levels

def common_depth(a, b):
    depth = 0
    for x, y in zip(a, b):
        if x != y:
            break
        depth += 1
    return depth

def code_sim(a, b):
    return 1.0 if a == b else IPC_LEVEL_WEIGHTS[common_depth(a, b)]

def technological_distance(q_codes, c_levels):
    if not q_codes or not c_levels:
        return 0.0
    q_to_c = sum(max(code_sim(q, c) for c in c_levels) for q in q_codes) / len(q_codes)
    c_to_q = sum(max(code_sim(c, q) for q in q_codes) for c in c_levels) / len(c_levels)
    return (q_to_c + c_to_q) / 2.0

levels = [[lv for lv in (ipc_levels(c) for c in cs) if lv] for cs in sub.codes]

def metrics(ranked_lists, relevant_sets, ks=(1, 5, 10, 50)):
    out = {}
    for k in ks:
        out[f"recall@{k}"] = float(np.mean([len(set(r[:k]) & rel) / len(rel) for r, rel in zip(ranked_lists, relevant_sets)]))
        out[f"hit@{k}"] = float(np.mean([bool(set(r[:k]) & rel) for r, rel in zip(ranked_lists, relevant_sets)]))
    rr = []
    for r, rel in zip(ranked_lists, relevant_sets):
        rank = next((i for i, pid in enumerate(r, 1) if pid in rel), None)
        rr.append(1.0 / rank if rank else 0.0)
    out["mrr"] = float(np.mean(rr))
    return out

In [ ]:
# Step 6: Encoders (FP16 on GPU)
import time
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

def embed_patent_bert(texts, batch_size=64):
    from transformers import BertModel, BertTokenizer
    tok = BertTokenizer.from_pretrained("anferico/bert-for-patents")
    model = BertModel.from_pretrained("anferico/bert-for-patents").to(device).eval()
    if device == "cuda":
        model = model.half()
    order = np.argsort([len(t) for t in texts])   # similar lengths per batch = less padding
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            idx = order[i:i + batch_size]
            enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            hidden = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            out[idx] = ((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).float().cpu().numpy()
    return out

def embed_sbert(name, texts):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(name, device=device)
    if device == "cuda":
        model.half()
    return model.encode(texts, batch_size=128, convert_to_numpy=True, show_progress_bar=True).astype(np.float32)

def embed(name, texts):
    start = time.time()
    vecs = embed_patent_bert(texts) if name == "patent-bert" else embed_sbert(name, texts)
    print(f"[{name}] {len(texts):,} patents in {(time.time() - start) / 60:.1f} min, dim {vecs.shape[1]}")
    return vecs

In [ ]:
# Step 7: Run the benchmark
import faiss

ids = sub.patent_id.tolist()
row = {pid: i for i, pid in enumerate(ids)}
q_rows = np.array([row[q] for q, _ in queries])
relevant = [rel for _, rel in queries]
q_codes = [[lv for lv in (ipc_levels(c) for c in sub.codes[i]) if lv] for i in q_rows]

results = []
for name in MODELS:
    vecs = embed(name, sub.meta.tolist())
    faiss.normalize_L2(vecs)
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    scores, nbrs = index.search(vecs[q_rows], POOL + 1)

    sd_lists, sdtd_lists = [], []
    for qi, qc, s_row, n_row in zip(q_rows, q_codes, scores, nbrs):
        cands = [(int(n), float(s)) for n, s in zip(n_row, s_row) if n != qi][:POOL]
        sd_lists.append([ids[n] for n, _ in cands])
        rescored = sorted(((n, (technological_distance(qc, levels[n]) + 1.0) * s / 2.0) for n, s in cands),
                          key=lambda x: x[1], reverse=True)
        sdtd_lists.append([ids[n] for n, _ in rescored])

    results.append({"encoder": name, "dim": int(vecs.shape[1]), "corpus": len(sub), "queries": len(queries),
                    "pool": POOL, "SD": metrics(sd_lists, relevant), "SDTD": metrics(sdtd_lists, relevant)})
    del vecs, index
    torch.cuda.empty_cache() if device == "cuda" else None

In [ ]:
# Step 8: Results and difference vs Patent-BERT
cols = ["recall@1", "recall@5", "recall@10", "hit@10", "recall@50", "mrr"]
table = pd.DataFrame([{"encoder": r["encoder"], "ranking": rank, **{c: r[rank][c] for c in cols}}
                      for r in results for rank in ("SD", "SDTD")])
print(f"Test set: {SUBSET}, {len(sub):,} patents, {len(queries)} queries, {sum(map(len, relevant)):,} citations\n")
display(table.round(4))

base = table[table.encoder == MODELS[0]].set_index("ranking")[cols]
diff = table[table.encoder != MODELS[0]].copy()
for c in cols:
    diff[c] = [v - base.loc[rk, c] for v, rk in zip(diff[c], diff.ranking)]
print(f"\nChange vs {MODELS[0]} (same ranking):")
display(diff.round(4))

In [ ]:
# Step 9: Save the results (add them to finetune/encoder_comparison.json in the project if you like)
import json
out = {"test_set": f"{SUBSET}{len(sub)}_s{SEED}_colab", "results": results}
with open("encoder_comparison_colab.json", "w") as f:
    json.dump(out, f, indent=2)
from google.colab import files
files.download("encoder_comparison_colab.json")